In [1]:
import requests # requests es una biblioteca de Python para hacer solicitudes HTTP de manera sencilla.
from bs4 import BeautifulSoup # BeautifulSoup es una biblioteca de Python para analizar documentos HTML y XML, facilitando la extracción de datos de páginas web.
from urllib.parse import urljoin # urljoin es una función de la biblioteca urllib.parse que se utiliza para construir URLs absolutas a partir de URLs relativas.
import sqlite3 
import time
import re# re es una biblioteca de Python para trabajar con expresiones regulares, que permiten buscar y manipular cadenas de texto de manera eficiente.
from functools import lru_cache # lru_cache es un decorador de Python que permite almacenar en caché los resultados de una función para mejorar su rendimiento, especialmente útil para funciones que se llaman con frecuencia con los mismos argumentos.
from concurrent.futures import ThreadPoolExecutor # ThreadPoolExecutor es una clase de la biblioteca concurrent.futures que permite ejecutar tareas de manera concurrente utilizando un pool de hilos, lo que puede mejorar el rendimiento en operaciones de E/S como las solicitudes HTTP.
from requests.adapters import HTTPAdapter # HTTPAdapter es una clase de la biblioteca requests que permite configurar cómo se manejan las conexiones HTTP, incluyendo la capacidad de implementar políticas de reintentos para mejorar la resiliencia de las solicitudes.
from urllib3.util.retry import Retry# Retry es una clase de la biblioteca urllib3 que se utiliza para definir políticas de reintentos en caso de fallos en las solicitudes HTTP, como errores de conexión o respuestas con códigos de estado específicos.
import pandas as pd

In [3]:
HEADERS = {'User-Agent': 'MiScraperEducativo/3.0 (contacto: tu@email.com)'}# Definimos un encabezado de agente de usuario personalizado para identificar nuestro scraper de manera ética y evitar bloqueos por parte del servidor.

In [2]:
def iniciar_db():
    conn = sqlite3.connect('biblioteca_total.db')
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")
    
    cursor.execute('''CREATE TABLE IF NOT EXISTS categories (
                        categoria_id INTEGER PRIMARY KEY AUTOINCREMENT,
                        nombre_categoria TEXT UNIQUE)''')
    
    cursor.execute('''CREATE TABLE IF NOT EXISTS authors (
                        id_autor INTEGER PRIMARY KEY AUTOINCREMENT,
                        nombre_autor TEXT,
                        fecha_nacimiento TEXT,
                        nacionalidad TEXT,
                        external_api_id TEXT,
                        total_trabajos_conocidos INTEGER,
                        api_source TEXT,
                        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    
    cursor.execute('''CREATE TABLE IF NOT EXISTS books (
                        id_libro INTEGER PRIMARY KEY AUTOINCREMENT,
                        titulo_libro TEXT,
                        precio REAL,
                        calificacion TEXT,
                        categoria_id INTEGER,
                        FOREIGN KEY(categoria_id) REFERENCES categories(categoria_id))''')
    
    cursor.execute('''CREATE TABLE IF NOT EXISTS book_author (
                        id_libro INTEGER,
                        id_autor INTEGER,
                        PRIMARY KEY(id_libro, id_autor),
                        FOREIGN KEY(id_libro) REFERENCES books(id_libro) ON DELETE CASCADE,
                        FOREIGN KEY(id_autor) REFERENCES authors(id_autor) ON DELETE CASCADE)''')
    conn.commit()
    print("✅ Base de datos inicializada y abierta.")
    return conn

DIAGRAMA UML

#REFERENCIAS DE DONDE CARAJO SAQUE LAS COSAS:
//<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="ALightintheAttic">ALight in the ...</a></h3>//TITULO EN h3 DENTRO DE a

//<p class="price_color">£51.77</p>//observemos la diferencia en como busca en el codigo directamente con comas le dice p, class_="price_color",en cambio titulo tenia un metadato, esto se observa como <h3> y <a> estan separados a diferencia de <p class> estan juntos//precio 


//<p class="instock availability">cantidad de disponibles

//<p class="star-rating Two"> calificacion

//obs el de la APi y del scraping son distintos el de la api es una funcion que devuelve el valor de la llave y el del scraping te el numero de conexion tipo 200=ok y 404=error 



In [3]:
# (Mantén la función crear_sesion_resiliente y limpiar_titulo de la respuesta anterior)
def crear_sesion_resiliente():
    session = requests.Session()# Creamos una sesión de requests para mantener conexiones persistentes y mejorar el rendimiento al hacer múltiples solicitudes al mismo servidor.
    # Importante: Añadimos un buen User-Agent porque las políticas de Wikimedia lo exigen
    session.headers.update({'User-Agent': 'MiScraperEducativo/3.0 (contacto: tu@email.com)'})# Configuramos reintentos para manejar mejor los errores temporales
    retries = Retry(total=3, backoff_factor=1, status_forcelist=[ 429, 500, 502, 503, 504 ])# Reintenta hasta 3 veces con un tiempo de espera exponencial entre intentos para ciertos códigos de error HTTP
    session.mount('https://', HTTPAdapter(max_retries=retries))# Aplicamos esta política de reintentos a todas las solicitudes HTTPS realizadas con esta sesión
    return session

http_session = crear_sesion_resiliente()# Creamos una sesión HTTP resiliente que se utilizará para todas las solicitudes a APIs externas, lo que mejora la estabilidad del scraper frente a errores temporales de red o del servidor.

def limpiar_titulo_basico(titulo):# Limpieza básica: solo quita paréntesis y su contenido
    if not titulo: return ""
    return re.sub(r'\([^)]*\)', '', str(titulo)).strip()

def limpiar_titulo_avanzado(titulo):# Limpieza avanzada: arregla codificación, quita volúmenes y subtítulos.
    """Limpieza agresiva: arregla codificación, quita volúmenes y subtítulos."""
    # Eliminamos pd.isna() y usamos solo 'if not titulo'
    if not titulo: return ""
    
    # Arreglar Mojibake (errores de codificación del scraping)
    t = str(titulo).replace("â€™", "'").replace("â€œ", '"').replace("â€", '"')
    
    # Quitar paréntesis (incluso anidados)
    t = re.sub(r'\(.*\)', '', t) 
    
    # Quitar "Vol. X" o "Volume X"
    t = re.sub(r'(?i),?\s*vol\.?\s*\d+.*', '', t)
    t = re.sub(r'(?i),?\s*volume\s*\d+.*', '', t)
    
    # Quedarse solo con el Título Principal (antes de los dos puntos)
    t_principal = t.split(':')[0]
    
    return t_principal.strip()

def buscar_en_openlibrary(titulo_limpio, datos_actuales):# Busca en OpenLibrary. Devuelve True si encontró autor.
    """Busca en OpenLibrary. Devuelve True si encontró autor."""
    try:# Intentamos buscar en OpenLibrary
        url_ol = "https://openlibrary.org/search.json"
        res_ol = http_session.get(url_ol, params={"title": titulo_limpio, "limit": 1}, timeout=7)
        
        if res_ol.status_code == 200:# Si la respuesta es exitosa, procesamos los datos
            data_ol = res_ol.json()# Verificamos si se encontraron resultados
            if data_ol.get("numFound", 0) > 0:# Si hay resultados, extraemos el primer resultado para obtener información del autor
                primer_resultado = data_ol["docs"][0]# Extraemos el nombre del autor y su ID de OpenLibrary
                autores = primer_resultado.get("author_name", []) # Si hay autores disponibles, actualizamos los datos actuales con la información obtenida
                
                if autores: # Si se encontró al menos un autor, actualizamos los datos actuales con la información obtenida
                    datos_actuales["nombre_real"] = autores[0] # Tomamos el primer autor encontrado como el nombre real del autor
                    datos_actuales["fuente"] = "OpenLibrary" # Indicamos que la fuente de esta información es OpenLibrary
                    datos_actuales["external_id"] = primer_resultado.get("author_key", ["N/A"])[0] # Guardamos el ID del autor en OpenLibrary para futuras consultas, o "N/A" si no está disponible
                    datos_actuales["total_obras"] = data_ol.get("numFound", 0) # Guardamos el total de obras encontradas para este título en OpenLibrary
                    
                    # Buscar fecha de nacimiento
                    if datos_actuales["external_id"] != "N/A":# Si tenemos un ID de autor válido, hacemos una consulta adicional para obtener la fecha de nacimiento
                        res_autor = http_session.get(f"https://openlibrary.org/authors/{datos_actuales['external_id']}.json", timeout=5)# Hacemos una solicitud a la API de OpenLibrary para obtener detalles del autor utilizando su ID
                        if res_autor.status_code == 200:# Si la respuesta es exitosa, extraemos la fecha de nacimiento del autor
                            datos_actuales["fecha_nacimiento"] = res_autor.json().get("birth_date", "Desconocido") # Guardamos la fecha de nacimiento del autor, o "Desconocido" si no está disponible
                    return True # Devolvemos True para indicar que se encontró información del autor en OpenLibrary
    except Exception as e:# Si ocurre cualquier error durante el proceso de búsqueda en OpenLibrary, lo silenciamos para que el scraper intente el siguiente método de búsqueda (Fallback)
        pass # Silenciamos el error para que intente el Fallback
    return False# Devolvemos False para indicar que no se encontró información del autor en OpenLibrary, lo que permitirá al scraper intentar el siguiente método de búsqueda (Fallback)


def buscar_en_wikipedia(titulo_limpio, datos_actuales):
    """Fallback: Busca en Wikipedia. Devuelve True si encontró autor."""
    try:# Intentamos buscar en Wikipedia como método de respaldo (Fallback) si OpenLibrary no tuvo éxito
        url_wiki = "https://es.wikipedia.org/api/rest_v1/page/summary/"# Hacemos una solicitud a la API de Wikipedia para buscar el título limpio
        res_wiki = http_session.get(url_wiki + titulo_limpio, timeout=7)# Si la respuesta es exitosa, procesamos los datos
        
        if res_wiki.status_code == 200:# Si la respuesta es exitosa, procesamos los datos
            data_wiki = res_wiki.json()# Verificamos si se encontraron resultados en Wikipedia y extraemos la información del autor si está disponible
            if "description" in data_wiki:# Si hay resultados, extraemos el primer resultado para obtener información del autor
                vol_info = data_wiki.get("description", {})# Extraemos el nombre del autor de Wikipedia, que suele estar en la lista "authors" dentro de "volumeInfo"
                autores_wiki = vol_info.get("authors", [])# Si hay autores disponibles, actualizamos los datos actuales con la información obtenida de Wikipedia
                
                if autores_wiki:
                    datos_actuales["nombre_real"] = autores_wiki[0]# Tomamos el primer autor encontrado como el nombre real del autor
                    datos_actuales["fecha_nacimiento"] = "No provista por Wikipedia"# Wikipedia no suele proporcionar la fecha de nacimiento del autor, por lo que indicamos que esta información no está disponible a través de esta fuente
                    datos_actuales["fuente"] = "Wikipedia (Fallback)"# Indicamos que la fuente de esta información es Wikipedia, y que se obtuvo como método de respaldo (Fallback) después de intentar OpenLibrary
                    datos_actuales["total_obras"] = data_wiki.get("totalItems", 0)# Guardamos el total de obras encontradas para este título en Wikipedia, lo que puede ser útil para futuras consultas o análisis
                    
                    return True# Devolvemos True para indicar que se encontró información del autor en Wikipedia, lo que significa que el método de respaldo (Fallback) tuvo éxito
    except Exception as e:# Si ocurre cualquier error durante el proceso de búsqueda en Wikipedia, lo silenciamos para que el scraper intente el siguiente método de búsqueda (Fallback)
        pass
    return False# Devolvemos False para indicar que no se encontró información del autor en Wikipedia, lo que permitirá al scraper intentar el siguiente método de búsqueda (Fallback) o concluir que no se pudo obtener información del autor.

def obtener_nacionalidad_wikidata(nombre_autor): 
    """Busca el país de origen (P27) en la web semántica de Wikidata."""
    url_base = "https://www.wikidata.org/w/api.php"# Definimos la URL base para las consultas a la API de Wikidata, que es una plataforma de datos estructurados que se utiliza para almacenar información sobre entidades como personas, lugares, eventos, etc. En este caso, utilizaremos Wikidata para obtener la nacionalidad de un autor a partir de su nombre.
    try:# Intentamos realizar la consulta a Wikidata para obtener la nacionalidad del autor
        # 1. Buscar Entidad
        res_search = http_session.get(url_base, params={"action": "wbsearchentities", "search": nombre_autor, "language": "es", "format": "json", "limit": 1}, timeout=5)
        data_search = res_search.json()
        if not data_search.get("search"): return "Desconocido"
        entity_id = data_search["search"][0]["id"]
        
        # 2. Buscar Propiedad P27
        res_claims = http_session.get(url_base, params={"action": "wbgetclaims", "entity": entity_id, "property": "P27", "format": "json"}, timeout=5)
        data_claims = res_claims.json()
        if "P27" not in data_claims.get("claims", {}): return "Desconocido"
        country_id = data_claims["claims"]["P27"][0]["mainsnak"]["datavalue"]["value"]["id"]
        
        # 3. Traducir Propiedad
        res_country = http_session.get(url_base, params={"action": "wbgetentities", "ids": country_id, "props": "labels", "languages": "es", "format": "json"}, timeout=5)
        data_country = res_country.json()
        
        return data_country["entities"][country_id]["labels"]["es"]["value"].capitalize()
    except:
        return "Desconocido"

In [4]:
@lru_cache(maxsize=1000)
def obtener_datos_autor(titulo_libro):
    """
    Función maestra corregida para evitar el KeyError: 'total_obras'
    """
    # Agregamos 'total_obras' aquí para que siempre exista
    datos = {
        "nombre_real": "Autor Desconocido",
        "fecha_nacimiento": "Desconocido", 
        "nacionalidad": "Desconocido", 
        "external_id": "N/A", 
        "total_obras": 0,  # <--- Esta es la llave que faltaba
        "fuente": "Ninguna"
    }
    
    titulo_basico = limpiar_titulo_basico(titulo_libro)
    
    encontrado = buscar_en_openlibrary(titulo_basico, datos)
    if not encontrado:
        encontrado = buscar_en_wikipedia(titulo_basico, datos)
        
    if not encontrado:
        titulo_agresivo = limpiar_titulo_avanzado(titulo_libro)
        if titulo_agresivo and titulo_agresivo != titulo_basico:
            encontrado = buscar_en_openlibrary(titulo_agresivo, datos)
            if not encontrado:
                encontrado = buscar_en_wikipedia(titulo_agresivo, datos)
            
            if encontrado:
                datos["fuente"] += " (Limpieza Agresiva)"

    if datos["nombre_real"] != "Autor Desconocido":
        datos["nacionalidad"] = obtener_nacionalidad_wikidata(datos["nombre_real"])

    return datos

In [5]:
def procesar_libro_individual(libro, id_cat):
    """
    Función auxiliar diseñada para correr dentro de un Hilo (Thread).
    Extrae los datos HTML del libro y llama a la API.
    """
    # Scraping defensivo de la tarjeta del libro
    titulo = libro.h3.a.get('title', 'Sin Título')
    
    precio_tag = libro.find('p', class_='price_color')
    precio_raw = precio_tag.text if precio_tag else "£0.00"
    precio = float(re.sub(r'[^\d.]', '', precio_raw))
    
    clases_p = libro.p.get('class', [])
    rating = clases_p[1] if len(clases_p) > 1 else "Sin_Calificacion"
    
    # Llamada a la API (Lenta, pero ahora paralela)
    meta = obtener_datos_autor(titulo)
    
    return {
        "titulo": titulo,
        "precio": precio,
        "rating": rating,
        "meta": meta,
        "id_cat": id_cat
    }

In [8]:
def ejecutar_sistema_completo():
    conn = iniciar_db()
    cursor = conn.cursor()
    base_url = 'https://books.toscrape.com/'
    
    # Sesión persistente para el scraping de la página principal
    session = requests.Session()
    session.headers.update(HEADERS)
    
    print("🚀 Iniciando Scraping Concurrente de Alto Rendimiento...")
    try:
        res_main = session.get(base_url, timeout=10)
        res_main.raise_for_status()
        sopa_main = BeautifulSoup(res_main.text, 'html.parser')
        cats = sopa_main.find('div', class_='side_categories').ul.li.ul.find_all('a')
    except Exception as e:
        print(f"❌ Error de conexión inicial: {e}")
        return

    # Iteramos por cada categoría
    for cat in cats:
        nombre_cat = cat.text.strip()
        
        # 🛑 CAMBIO CLAVE 1: Insertar categoría y hacer COMMIT inmediato
        # Esto evita el error "FOREIGN KEY constraint failed"
        cursor.execute("INSERT OR IGNORE INTO categories (nombre_categoria) VALUES (?)", (nombre_cat,))
        conn.commit() 
        
        cursor.execute("SELECT categoria_id FROM categories WHERE nombre_categoria = ?", (nombre_cat,))
        id_cat = cursor.fetchone()[0]
        
        url_actual = urljoin(base_url, cat['href'])
        
        # Paginación dentro de la categoría
        while url_actual:
            print(f"Procesando categoría: {nombre_cat} -> {url_actual}")
            try:
                r_pag = session.get(url_actual, timeout=10)
                sopa_pag = BeautifulSoup(r_pag.text, 'html.parser')
                libros = sopa_pag.find_all('article', class_='product_pod')
                
                with ThreadPoolExecutor(max_workers=5) as executor:
                    resultados_pagina = list(executor.map(lambda l: procesar_libro_individual(l, id_cat), libros))
                
                # 🛑 CAMBIO CLAVE 2: SE ELIMINÓ LA LÍNEA cursor.execute("BEGIN TRANSACTION;")
                # Python iniciará la transacción implícitamente sin generar el error "cannot start a transaction"
                
                for dato in resultados_pagina:
                    # 1. Insertar Libro
                    cursor.execute("INSERT INTO books (titulo_libro, precio, calificacion, categoria_id) VALUES (?,?,?,?)", 
                                   (dato["titulo"], dato["precio"], dato["rating"], dato["id_cat"]))
                    id_libro = cursor.lastrowid
                    
                    # 2. Insertar Autor 
                    meta = dato["meta"]
                    nombre_autor_final = meta["nombre_real"]
                    
                    cursor.execute('''INSERT OR IGNORE INTO authors 
                        (nombre_autor, fecha_nacimiento, nacionalidad, external_api_id, total_trabajos_conocidos, api_source)
                        VALUES (?, ?, ?, ?, ?, ?)''', 
                        (nombre_autor_final, meta["fecha_nacimiento"], meta["nacionalidad"], 
                         str(meta["external_id"]), meta["total_obras"], meta["fuente"]))
                    
                    # 3. Recuperar ID del Autor y Vincular
                    cursor.execute("SELECT id_autor FROM authors WHERE nombre_autor = ?", (nombre_autor_final,))
                    res_autor = cursor.fetchone()
                    
                    if res_autor:
                        cursor.execute("INSERT OR IGNORE INTO book_author (id_libro, id_autor) VALUES (?,?)", 
                                       (id_libro, res_autor[0]))
                
                # Commit masivo de toda la página
                conn.commit() 
                
            except Exception as page_err:
                print(f"❌ Error procesando la página {url_actual}: {page_err}")
                conn.rollback() # Si falla, se deshacen solo los libros de ESTA página
                
            # Control de Paginación Defensivo
            next_tag = sopa_pag.find('li', class_='next')
            if next_tag and next_tag.find('a'):
                url_actual = urljoin(url_actual, next_tag.a.get('href', ''))
            else:
                url_actual = None
                
            time.sleep(0.5) 
        
        print(f"✅ Categoría '{nombre_cat}' finalizada.")

    conn.close()
    print("✨ Proceso completado exitosamente.")

if __name__ == "__main__":
    ejecutar_sistema_completo()

✅ Base de datos inicializada y abierta.
🚀 Iniciando Scraping Concurrente de Alto Rendimiento...
Procesando categoría: Travel -> https://books.toscrape.com/catalogue/category/books/travel_2/index.html
✅ Categoría 'Travel' finalizada.
Procesando categoría: Mystery -> https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Procesando categoría: Mystery -> https://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html
✅ Categoría 'Mystery' finalizada.
Procesando categoría: Historical Fiction -> https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Procesando categoría: Historical Fiction -> https://books.toscrape.com/catalogue/category/books/historical-fiction_4/page-2.html
✅ Categoría 'Historical Fiction' finalizada.
Procesando categoría: Sequential Art -> https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
Procesando categoría: Sequential Art -> https://books.toscrape.com/catalogue/category/books/sequen

In [4]:
def ejecutar_consultas_emocion():
    conn = sqlite3.connect('biblioteca_total.db')
    
    # Diccionario para convertir el texto de calificación a número en SQL
    mapping_calificacion = "CASE calificacion WHEN 'One' THEN 1 WHEN 'Two' THEN 2 WHEN 'Three' THEN 3 WHEN 'Four' THEN 4 WHEN 'Five' THEN 5 ELSE 0 END"

    print("--- EJECUTANDO CONSULTAS REQUERIDAS ---")

    # 1. Libros con más de 3 estrellas por menos de £10
    query1 = f"SELECT titulo_libro, precio, calificacion FROM books WHERE ({mapping_calificacion}) > 3 AND precio < 10"
    print("\n1. Libros con > 3 estrellas y precio < £10:")
    print(pd.read_sql_query(query1, conn))

    # 2. Autor con peor promedio de rating (mínimo 1 libro en este scraping, ajustable)
    query2 = f"""
        SELECT a.nombre_autor, AVG({mapping_calificacion}) as promedio_rating
        FROM authors a
        JOIN book_author ba ON a.id_autor = ba.id_autor
        JOIN books b ON ba.id_libro = b.id_libro
        GROUP BY a.id_autor
        HAVING COUNT(b.id_libro) >= 1
        ORDER BY promedio_rating ASC
        LIMIT 1
    """
    print("\n2. Autor con peor promedio de rating:")
    print(pd.read_sql_query(query2, conn))

    # 3. Categoría con mayor precio promedio
    query3 = """
        SELECT c.nombre_categoria, AVG(b.precio) as precio_promedio
        FROM categories c
        JOIN books b ON c.categoria_id = b.categoria_id
        GROUP BY c.categoria_id
        ORDER BY precio_promedio DESC
        LIMIT 1
    """
    print("\n3. Categoría con mayor precio promedio:")
    print(pd.read_sql_query(query3, conn))

    # 4. Top 5 autores con más libros
    query4 = """
        SELECT a.nombre_autor, COUNT(ba.id_libro) as total_libros
        FROM authors a
        JOIN book_author ba ON a.id_autor = ba.id_autor
        GROUP BY a.id_autor
        ORDER BY total_libros DESC
        LIMIT 5
    """
    print("\n4. Top 5 autores con más libros:")
    print(pd.read_sql_query(query4, conn))

    # 5. CONSULTA OBLIGATORIA POR PAÍS: ¿Qué país produce más libros con rating > 3 estrellas?
    query5 = f"""
        SELECT a.nacionalidad as pais, COUNT(b.id_libro) as total_libros_buenos
        FROM authors a
        JOIN book_author ba ON a.id_autor = ba.id_autor
        JOIN books b ON ba.id_libro = b.id_libro
        WHERE ({mapping_calificacion}) > 3
        GROUP BY a.nacionalidad
        ORDER BY total_libros_buenos DESC
        LIMIT 1
    """
    print("\n5. (OBLIGATORIA) País con más libros de rating > 3:")
    print(pd.read_sql_query(query5, conn))

    conn.close()

ejecutar_consultas_emocion()

--- EJECUTANDO CONSULTAS REQUERIDAS ---

1. Libros con > 3 estrellas y precio < £10:
Empty DataFrame
Columns: [titulo_libro, precio, calificacion]
Index: []

2. Autor con peor promedio de rating:
   nombre_autor  promedio_rating
0  Paul Theroux              1.0

3. Categoría con mayor precio promedio:
  nombre_categoria  precio_promedio
0         Suspense            58.33

4. Top 5 autores con más libros:
        nombre_autor  total_libros
0  Autor Desconocido            27
1       Stephen King            12
2        Worth Books             9
3               高屋奈月             9
4    Sophie Kinsella             8

5. (OBLIGATORIA) País con más libros de rating > 3:
             pais  total_libros_buenos
0  Estados unidos                  165


In [5]:
def prueba_performance_indexacion():
    conn = sqlite3.connect('biblioteca_total.db')
    cursor = conn.cursor()

    # Consulta de prueba: buscar libros en un rango de precio específico
    query_busqueda = "SELECT * FROM books WHERE precio BETWEEN 20.0 AND 25.0"

    print("--- PRUEBA DE RENDIMIENTO ---")

    # 1. Medir ANTES del índice
    inicio = time.time()
    for _ in range(100): # Repetimos para notar la diferencia en milisegundos
        cursor.execute(query_busqueda).fetchall()
    fin = time.time()
    tiempo_sin_indice = fin - inicio
    print(f"Tiempo promedio sin índice: {tiempo_sin_indice:.5f} segundos")

    # 2. Crear el índice
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_precio ON books(precio)")
    conn.commit()
    print("✅ Índice 'idx_precio' creado.")

    # 3. Medir DESPUÉS del índice
    inicio = time.time()
    for _ in range(100):
        cursor.execute(query_busqueda).fetchall()
    fin = time.time()
    tiempo_con_indice = fin - inicio
    print(f"Tiempo promedio con índice: {tiempo_con_indice:.5f} segundos")

    # Explicación del resultado
    mejora = ((tiempo_sin_indice - tiempo_con_indice) / tiempo_sin_indice) * 100
    print(f"\nResultados: La búsqueda es un {mejora:.2f}% más rápida con el índice.")
    print("Explicación: Sin el índice, SQLite realiza un 'Full Table Scan'. Con el índice, usa una estructura B-Tree para saltar directamente a los registros.")

    conn.close()

prueba_performance_indexacion()

--- PRUEBA DE RENDIMIENTO ---
Tiempo promedio sin índice: 0.04742 segundos
✅ Índice 'idx_precio' creado.
Tiempo promedio con índice: 0.03310 segundos

Resultados: La búsqueda es un 30.20% más rápida con el índice.
Explicación: Sin el índice, SQLite realiza un 'Full Table Scan'. Con el índice, usa una estructura B-Tree para saltar directamente a los registros.
